### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

  Using cached numpy-2.4.3-cp314-cp314-macosx_14_0_x86_64.whl.metadata (6.6 kB)
  Using cached scikit_learn-1.8.0-cp314-cp314-macosx_10_15_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp314-cp314-macosx_14_0_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached numpy-2.4.3-cp314-cp314-macosx_14_0_x86_64.whl (6.5 MB)
Using cached scikit_learn-1.8.0-cp314-cp314-macosx_10_15_x86_64.whl (8.5 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp314-cp314-macosx_14_0_x86_64.whl (22.7 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [10]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ..., 11306,  1828, 10791], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.


In [29]:
import numpy as np

# Elijo 5 documentos al azar
rng = np.random.default_rng(42)
muestra_idx = rng.choice(X_train.shape[0], size=5, replace=False)

# Obtengo la similaridad coseno 
muestra_docs = X_train[muestra_idx]
cossim = cosine_similarity(muestra_docs, X_train)  # shape: (5, n_docs)

for i, idx in enumerate(muestra_idx):
    print(f"Documento {i+1}:")
    print(f"Índice {idx}, Etiqueta: {newsgroups_train.target_names[newsgroups_train.target[idx]]}")
    print(f"Contenido: {newsgroups_train.data[idx][:200]} ... \n")
    
    # Top 5 similares
    sim_indices = np.argsort(cossim[i])[::-1][1:6]  

    print("Documentos más similares:")
    for sim_idx in sim_indices:
        print(f" - Índice {sim_idx}, Similaridad: {cossim[i][sim_idx]:.4f}, Etiqueta: {newsgroups_train.target_names[newsgroups_train.target[sim_idx]]}")
        print(f"   Contenido: {newsgroups_train.data[sim_idx][:200]} ...\n")
    print("\n" + "="*80 + "\n")

Documento 1:
Índice 8754, Etiqueta: talk.religion.misc
Contenido: 
/(hudson)
/If someone inflicts pain on themselves, whether they enjoy it or not, they
/are hurting themselves.  They may be permanently damaging their body.

That is true.  It is also none of your bu ... 

Documentos más similares:
 - Índice 6552, Similaridad: 0.4904, Etiqueta: talk.religion.misc
   Contenido: 
If I have a habit that I really want to break, and I am willing to
make whatever sacrifice I need to make to break it, then I do so.
There have been bad habits of mine that I've decided to put forth  ...

 - Índice 10613, Similaridad: 0.4812, Etiqueta: talk.religion.misc
   Contenido: /(hudson)
/Yes you do.  Who is to say that it is immoral for onesself to experience
/pain or to be hurt in some other way.  Maybe unpleasant, but that doesn't
/say anything about morality.

It violate ...

 - Índice 3616, Similaridad: 0.4653, Etiqueta: talk.religion.misc
   Contenido: 
And I maintain:

Some people do not want to ent

Analizando los resultados puede verse que en todos los casos las etiquetas y texto de los documentos similares tienen mucho sentido.


**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.


In [ ]:
# Saco similaridad coseno
cossim = cosine_similarity(X_test, X_train)

# Obtengo el documento más similar del grupo de entrenamiento para cada documento del grupo de test
best_match_idx = np.argmax(cossim, axis=1)

# NOTA: Asumo que la consigna se refiere a asignar la etiqueta al grupo de test, ya que se pide construir un modelo de clasificacion
y_pred = y_train[best_match_idx]

# Calculo el f1 score macro
f1_score(y_test, y_pred, average='macro')

0.5049911553681621

Se obtiene un f1_score similar a anteriores ejercicios.


**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.


In [83]:
tfidfvect = TfidfVectorizer(binary=True, smooth_idf=False, norm=None)
X_train = tfidfvect.fit_transform(newsgroups_train.data)

nb_multinomial = MultinomialNB()
nb_multinomial.fit(X_train, y_train)

y_pred = nb_multinomial.predict(X_test)
print(f"MultinomialNB: {f1_score(y_test, y_pred, average='macro'):.2f}")

nb_complement = ComplementNB()
nb_complement.fit(X_train, y_train)

y_pred = nb_complement.predict(X_test)
print(f"ComplementNB: {f1_score(y_test, y_pred, average='macro'):.2f}")

MultinomialNB: 0.63
ComplementNB: 0.64


Mediante la prueba de distintos hiperparametros en la instanciacion del vectorizador, di con esta configuración que igualó el desempeño de ambos modelos de Naive Bayes en un numero relativamente alto, por ende decidí continuar con ella. Ademas, hay un aumento del f1_score respecto de anteriores ejercicios.


**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


In [93]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}
term_doc = X_train.T

palabras = ['oil', 'blood', 'freeze', 'bill', 'boots']
idx_palabras = [tfidfvect.vocabulary_[p] for p in palabras]

cossim = cosine_similarity(term_doc[idx_palabras], term_doc)

for i, palabra in enumerate(palabras):
    print(f"Palabra: '{palabra}'")
    sim_indices = np.argsort(cossim[i])[::-1][1:6] # Top 5 similares
    print("Palabras más similares:")
    print(f", ".join([idx2word[idx] for idx in sim_indices]))
    print("\n")


Palabra: 'oil'
Palabras más similares:
chromed, kuwaitis, octane, pistons, valve


Palabra: 'blood'
Palabras más similares:
armenian, glucose, caucasus, armenians, procuracy


Palabra: 'freeze'
Palabras más similares:
kamloops, amarillo, hanson, miron, albuquerque


Palabra: 'bill'
Palabras más similares:
clinton, senate, congress, voted, rights


Palabra: 'boots'
Palabras más similares:
tankers, gloves, velcro, deepest, jeans




Las palabras fueron elegidas para rapida verificación del contexto de las palabras similares, y en la mayoría de los casos guardan relación con las palabras elegidas.